# Predikcija cijene polovnih automobila - Regresija

## Cilj projekta je razviti model mašinskog učenja koji predviđa vrijednost kolone 'priceUSD'. Radi se o regresionom problemu. Model predviđa cijenu, numeričku vrijednost, a ne klasu ili kategoriju.

## U projektu će biti sprovedeni sljedeći koraci: istraživanje podataka, čišćenje, inženjering karakteristika, pretprocesiranje, treniranje više modela, evaluacija pomoću regresionih metrika i izbor najboljeg modela.

# Učitavanje podataka

In [1]:
from pathlib import Path 
import pandas as pd
DATA_PATH  = Path("../data/cars.csv")

In [2]:
df = pd.read_csv(DATA_PATH)

# Prikazivanje prvih nekoliko redova

In [3]:
df.head()

,make,model,priceUSD,year,condition,mileage(kilometers),fuel_type,volume(cm3),color,transmission,drive_unit,segment
0,mazda,2,5500,2008,with mileage,162000.0,petrol,1500.0,burgundy,mechanics,front-wheel drive,B
1,mazda,2,5350,2009,with mileage,120000.0,petrol,1300.0,black,mechanics,front-wheel drive,B
2,mazda,2,7000,2009,with mileage,61000.0,petrol,1500.0,silver,auto,front-wheel drive,B
3,mazda,2,3300,2003,with mileage,265000.0,diesel,1400.0,white,mechanics,front-wheel drive,B
4,mazda,2,5200,2008,with mileage,97183.0,diesel,1400.0,gray,mechanics,front-wheel drive,B


## Uvidom u učitane podatke može se uočiti da tabela ima 12 kolona i to:

* 'make' - data marka automobila;
* 'model' - model automobila;
* 'priceUSD' - cijena automobila u dolarima;
* 'year' - godina proizvodnje;
* 'condition' - stanje automobila;
* 'mileage(kilometers)' - pređena kilometraža;
* 'fuel_type' - vrsta goriva;
* 'volume(cm3) - zapremina motora;
* 'color' - boja automobila;
* 'transmission' - tip mjenjača;
* 'drive_unit' - tip pogona;
* 'segment' - klasa automobila.

Ciljna promjenljiva je 'priceUSD'.

# Prikaz nasumičnih redova

In [4]:
df.sample(5, random_state=42)

,make,model,priceUSD,year,condition,mileage(kilometers),fuel_type,volume(cm3),color,transmission,drive_unit,segment
19270,mitsubishi,carisma,2050,1999,with mileage,250000.0,petrol,1800.0,blue,mechanics,front-wheel drive,M
25927,ford,fusion,4500,2006,with mileage,160000.0,petrol,1400.0,burgundy,mechanics,front-wheel drive,M
23388,mercedes-benz,e-klass,3500,1999,with mileage,485000.0,diesel,2900.0,blue,mechanics,rear drive,E
53189,bmw,x3,21000,2013,with mileage,159000.0,diesel,2000.0,black,auto,NaN,J
34058,renault,megane,3990,2002,with mileage,331700.0,diesel,1900.0,other,mechanics,front-wheel drive,C


# Osnovna struktura skupa podataka

In [5]:
df.shape

(56244, 12)

## Broj redova i kolona.

In [6]:
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

Rows: 56244
Columns: 12


## Pregled kolona.

In [7]:
df.columns.to_list()

['make',
 'model',
 'priceUSD',
 'year',
 'condition',
 'mileage(kilometers)',
 'fuel_type',
 'volume(cm3)',
 'color',
 'transmission',
 'drive_unit',
 'segment']

## U narednom koraku će biti prikazani:
 
* nazivi kolona;
* broj vrijednosti koje nisu nedostajuće;
* tip podataka za svaku kolonu;
* informacije o memoriji.

In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 56244 entries, 0 to 56243
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   make                 56244 non-null  str    
 1   model                56244 non-null  str    
 2   priceUSD             56244 non-null  int64  
 3   year                 56244 non-null  int64  
 4   condition            56244 non-null  str    
 5   mileage(kilometers)  56244 non-null  float64
 6   fuel_type            56244 non-null  str    
 7   volume(cm3)          56197 non-null  float64
 8   color                56244 non-null  str    
 9   transmission         56244 non-null  str    
 10  drive_unit           54339 non-null  str    
 11  segment              50953 non-null  str    
dtypes: float64(2), int64(2), str(8)
memory usage: 5.1 MB


## Kod kolone 'priceUSD' koja predstavlja ciljnu promjenljivu, nema nedostajućih vrijednosti.

# Važan korak - provjera nedostajućih vrijednosti

In [9]:
missing_values = df.isna().sum()
missing_values[missing_values > 0]


volume(cm3)      47
drive_unit     1905
segment        5291
dtype: int64

## Ispitivanje da li se nedostajuće vrijednosti pojavljuju kao tekst.

In [10]:
missing_like_values = ["NaN", "nan", "NULL", "null", "None", "none", "", " "]

for column in df.columns:
    if df[column].dtype == "object":
        count = df[column].astype(str).str.strip().isin(missing_like_values).sum()

        if count > 0:
            print(f"{column}: {count}")

# Analiza ciljne promjenljive
U ovom projektu ciljna promjenljiva je priceUSD.

In [11]:
df["priceUSD"].head()

0    5500
1    5350
2    7000
3    3300
4    5200
Name: priceUSD, dtype: int64

## Tip podatka ciljne promjenljive.

In [12]:
df["priceUSD"].dtype

dtype('int64')

## Broj jedinstvenih vrijednosti i prikaz prvih deset vrijednosti.

In [13]:
print(df["priceUSD"].nunique())
df["priceUSD"].unique()[:10]

2970


array([5500, 5350, 7000, 3300, 5200, 3400, 5000, 7300, 6400, 6132])

# Analiza numeričkih kolona 
Kolone za koje se očekuje da budu numeričke.

In [14]:
numeric_columns = [
    "year",
    "mileage(kilometers)",
    "volume(cm3)"
]
df[numeric_columns].dtypes

year                     int64
mileage(kilometers)    float64
volume(cm3)            float64
dtype: object

## Statistički podaci o numeričkim kolonama.

In [15]:
df[numeric_columns].describe()

,year,mileage(kilometers),volume(cm3)
count,56244.000000,5.624400e+04,56197.000000
mean,2003.454840,2.443956e+05,2104.860615
std,8.144247,3.210307e+05,959.201633
min,1910.000000,0.000000e+00,500.000000
25%,1998.000000,1.370000e+05,1600.000000
50%,2004.000000,2.285000e+05,1996.000000
75%,2010.000000,3.100000e+05,2300.000000
max,2019.000000,9.999999e+06,20000.000000


## U koloni koja se odnosi na godinu proizvodnje automobila ('year'), najstariji auto je iz 1910, a najnoviji iz 2019. godine. Moguće je da postoji takvi stari auti. Na primjer, oldtimeri. U koloni koja se odnosi na kilometražu ('mileage(kilometers)') neke vrijednosti nisu realne. Tako imamo aute koji nisu prešli ni jedan kilometar, a radi se o polovnim autima. I imamo podatak da je maksimalna kilometraža 9999999 km, što takođe nije realan podatak. Moguće je da je prilikom unosa napravljena greška. U koloni koja se odnosi na zapreminu motora ('volume(cm3)'), za max zapreminu motora je navedena vrijednost od 20000 cm3, što nije realno. 


# Analiza kategorijskih kolona

U ovom skupu podataka, to su kolone kao što su:
* 'make'
* 'model'
* 'condition'
* 'fuel_type'
* 'color'
* 'transmission'
* 'drive_unit'
* 'segment'

In [16]:
categorical_columns = [
    "make",
    "model",
    "condition",
    "fuel_type",
    "color",
    "transmission",
    "drive_unit",
    "segment"
]

df[categorical_columns].head()

,make,model,condition,fuel_type,color,transmission,drive_unit,segment
0,mazda,2,with mileage,petrol,burgundy,mechanics,front-wheel drive,B
1,mazda,2,with mileage,petrol,black,mechanics,front-wheel drive,B
2,mazda,2,with mileage,petrol,silver,auto,front-wheel drive,B
3,mazda,2,with mileage,diesel,white,mechanics,front-wheel drive,B
4,mazda,2,with mileage,diesel,gray,mechanics,front-wheel drive,B


# Broj jedinstvenih vrijednosti

In [17]:
for column in categorical_columns:
    print(column)
    print(df[column].nunique())
    print("-" * 40)

make
96
----------------------------------------
model
1034
----------------------------------------
condition
3
----------------------------------------
fuel_type
3
----------------------------------------
color
13
----------------------------------------
transmission
2
----------------------------------------
drive_unit
4
----------------------------------------
segment
9
----------------------------------------


# Broj jedinstvenih vrijednosti za svaku kolonu

In [18]:
for column in categorical_columns:
    print(column)
    print(df[column].value_counts(dropna=False))
    print("-" * 40)

make
make
volkswagen    6861
audi          4030
bmw           4013
opel          3779
renault       3713
              ... 
trabant          1
jac              1
asia             1
tagaz            1
saipa            1
Name: count, Length: 96, dtype: int64
----------------------------------------
model
model
passat      2086
5-seriya    1476
a6          1276
golf        1070
astra       1013
            ... 
xb             1
xc40           1
xjs            1
xt5            1
z3             1
Name: count, Length: 1034, dtype: int64
----------------------------------------
condition
condition
with mileage    55278
with damage       512
for parts         454
Name: count, dtype: int64
----------------------------------------
fuel_type
fuel_type
petrol        36405
diesel        19792
electrocar       47
Name: count, dtype: int64
----------------------------------------
color
color
black       12385
silver      10075
blue         8083
gray         5807
white        5292
green        3911
ot

## Kolone 'drive_unit' i 'segment' imaju nedostajuće vrijednosti.

## Ekstremne vrijednosti 

Potrebno ih je analizirati jer mogu da iskrive model, mogu da ukažu na greške u podacima. Ukoliko se ne isključe mogu dovesti do izvođenja lažnih zaključaka. Osim toga, ukazuju na postojanje nekih specifičnih slučajeva, na primjer postojanje oldtimera.

## Ispitaće se kolone :
* priceUSD 
* year
* mileage(kilometers)
* volume(cm3)

## Ispitivanje kolone sa cijenom automobila

In [19]:
df["priceUSD"].describe()

count     56244.000000
mean       7415.456440
std        8316.959261
min          48.000000
25%        2350.000000
50%        5350.000000
75%        9807.500000
max      235235.000000
Name: priceUSD, dtype: float64

# Cijene su u normalnim granicama. Moguće je da se prodaje samo neki rezervni dio, ili je u pitanju prodaja polovnog luksuznog automobila.

## Analiza kolone - godina proizvodnje

In [20]:
df["year"].describe()

count    56244.000000
mean      2003.454840
std          8.144247
min       1910.000000
25%       1998.000000
50%       2004.000000
75%       2010.000000
max       2019.000000
Name: year, dtype: float64

## Godine realne. Mogu se prodavati old timeri iz 1910.

## Analiza kolone 'mileage(kilometers)'

In [21]:
df['mileage(kilometers)'].describe()

count    5.624400e+04
mean     2.443956e+05
std      3.210307e+05
min      0.000000e+00
25%      1.370000e+05
50%      2.285000e+05
75%      3.100000e+05
max      9.999999e+06
Name: mileage(kilometers), dtype: float64

## Imamo nerealnu vrijednost kod kilometraže kod max koja iznosi 9999999 km. Nije moguća.

## Koliko imamo redova sa ovom vrijednošću

In [22]:
(df["mileage(kilometers)"] == 9999999).sum()

np.int64(22)

## Ispitivanje koliko ima redova s 0 pređenih kilometara.

In [23]:
(df["mileage(kilometers)"]==0).sum()

np.int64(190)

## Provjera godišta automobila za koja je podatak od 0 pređenih kilometara.

In [24]:
df[df["mileage(kilometers)"]==0]  [["year"]].head(10)

,year
184,2007
809,1991
863,1961
910,1992
1009,1994
1059,1966
1069,1962
1376,1988
1576,1989
1836,1993


## Nerealno da stariji automobili imaju kilometražu 0 km.

## Analiza kolone 'volume(cm3)'

In [25]:
df['volume(cm3)'].describe()

count    56197.000000
mean      2104.860615
std        959.201633
min        500.000000
25%       1600.000000
50%       1996.000000
75%       2300.000000
max      20000.000000
Name: volume(cm3), dtype: float64

## Greška kod max vrijednosti

In [26]:
df[df['volume(cm3)'] > 7000][["make", "year"]].head(10)

,make,year
81,mazda,2006
354,mazda,2003
929,renault,1990
1537,audi,1993
1768,audi,1992
2144,audi,1984
2479,rover,1995
2587,peugeot,2001
3411,peugeot,2010
3714,mazda,1997


## Zapremina motora preko 7000 cm3 je veoma rijetka. Moguće da je napravljena greška prilikom unosa.

## Zaključci

Dataset sadrži 56244 zapisa o polovnim automobilima i 12 kolona s karakteristikama o tim kolonama.

Prije modeliranja treba uraditi sljedeće:

* standardizovati nazive kolona;
* ukloniti višak razmaka iz teksta;
* izvršiti standardizaciju nedostajućih vrijednosti;
* konverzija numeričkih kolona zbog pouzdanosti i mogućeg kasnijeg korištenja.
* standardizacija kategorijskih vrijednosti
* uklanjanje nevalidnih redova kod kolona 'year', 'mileage(kilometers)', 'volume(cm3)'.
* biće urađen inženjering karakteristika; nove kolone će biti: 'car_age' koja će opisivati    
 starost   automobila; kolona 'mileage_per_year' u kojoj će biti prosječna kilometraža po godini starosti; kolona 'engine_volume_liters' u kome će umjesto zapremine motora u cm3 biti data zapremina motora u litrima; kolona 'is_newer_car' s indikatorom da li je automobil novijeg godišta; kolona 'is_high_mileage' s indikatorom da li automobil ima veliku kilometražu i kolona 'brand_model' u kojoj će biti kombinacija marke i modela.
* Nakon dobijenog očišćenog seta nad kojim je urađen i inženjering karakteristika biće izvršeno  
  pretprocesiranje, treniranje modela, upoređivanje regresionih modela i izbor modela koji daje najbolji rezultat na osnovu metrika.


